### Imports

In [44]:
# Imports
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
# from pinecone import Pinecone
# from langchain_pinecone import PineconeVectorStore
# from pinecone import ServerlessSpec
from langchain_ollama import OllamaLLM



### Step 1: Document Ingestion & Indexing
- Parse both PDFs into clean text.
- Generate embeddings using an open-source model.
- Store in a vector database (e.g., FAISS, Chroma).
- Preserve metadata: document, section, page number.

In [2]:
# Create a function to load and split the PDF documents
def load_n_split_pdf(path: str) -> list:
    loader = PyPDFLoader(path)
    docs = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = text_splitter.split_documents(docs)
    return chunks

In [3]:
# Load amd Split Apple & Tesla data
docs_apple = load_n_split_pdf("../documents/10-Q4-2024-As-Filed.pdf")
docs_tesla = load_n_split_pdf("../documents/tsla-20231231-gen.pdf")

In [4]:
# docs sizes
len(docs_apple), len(docs_tesla)

(499, 533)

In [34]:
# Create Embeddings and FAISS Vector Store
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

embeddings = OllamaEmbeddings(model="mxbai-embed-large")
index = faiss.IndexFlatL2(len(embeddings.embed_query("test")))
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [35]:
print(vector_store.index.ntotal)

0


In [36]:
# Add Apple documents to Pinecone index
from uuid import uuid4

print("Adding Apple documents to FAISS Started...")
uuids_apple = [str(uuid4()) for _ in range(len(docs_apple))]
vector_store.add_documents(documents=docs_apple, ids=uuids_apple)
print("Adding Apple documents to FAISS Completed...")

Adding Apple documents to FAISS Started...
Adding Apple documents to FAISS Completed...


In [37]:
# Add Tesla documents to Pinecone index
print("Adding Tesla documents to FAISS Started...")
uuids_tesla = [str(uuid4()) for _ in range(len(docs_tesla))]
vector_store.add_documents(documents=docs_tesla, ids=uuids_tesla)
print("Adding Tesla documents to FAISS Completed...")

Adding Tesla documents to FAISS Started...
Adding Tesla documents to FAISS Completed...


In [38]:
print(vector_store.index.ntotal)

1032


### Step 2: Build a Retrieval Pipeline \n
- Implement vector similarity search to retrieve top-5 relevant chunks.
- Add a re-ranker.
- Justify your choice in the design report

In [39]:
def retrieve_candidates(query: str, k: int = 20):
    docs = vector_store.similarity_search(query, k=k)
    return docs

In [40]:
# Reranking function using Ollama LLM
def rerank_documents(query: str, docs, top_n: int = 5):
    reranker_model = "qllama/bge-reranker-v2-m3"
    reranker = OllamaLLM(model=reranker_model, temperature=0)
    scored_docs=[]
    for doc in docs:
        prompt = f"Query: {query}\nDocument: {doc.page_content}"
        score = reranker.invoke(prompt)
        # Model usually returns a float score
        try:
            score = float(score.strip())
        except:
            score = 0.0
        scored_docs.append((doc, score))
    # Sort descending
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in scored_docs[:top_n]]

In [42]:
def retrieval_pipeline(query: str):
    # Step 1: Retrieve candidates
    candidates = retrieve_candidates(query, k=20)

    # Step 2: Rerank
    top_docs = rerank_documents(query, candidates, top_n=5)

    return top_docs

In [45]:
retrieval_pipeline("what is the compensation received by any of the registrant’s executive officers during the relevant recovery period?")

ResponseError: model 'qllama/bge-reranker-v2-m3' not found (status code: 404)

### Step 3: LLM Integration
- Use a locally hosted or open-access LLM (e.g., Llama 3, Mistral, Phi-3).
- Do not use GPT-4, Claude, or any closed API.
- Design a custom prompt that:
    - Uses only retrieved context.
    - Cites sources as ["Apple 10-K", "Item 8", "p. 28"].
    - Responds with "Not specified in the document." if the answer is not in the provided files.
    - Refuses out-of-scope questions with: "This question cannot be answered based on the provided documents."

In [44]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3",
    temperature=0  # critical for factual grounding
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a financial document QA assistant.

You MUST follow these rules strictly:

1. Answer ONLY using the provided context.
2. Do NOT use external knowledge.
3. Every factual statement must be cited in this format:
   ["document", "section", "page number"].
4. If the answer is not explicitly found in the context, respond exactly:
   "Not specified in the document."
5. If the question is unrelated to the provided financial documents, respond exactly:
   "This question cannot be answered based on the provided documents."

----------------------
Context:
{context}
----------------------

Question:
{question}

Answer:
""")

In [47]:
def format_context(docs):
    formatted_chunks = []

    for doc in docs:
        citation = f'["{doc.metadata.get("document_name")}", "{doc.metadata.get("section")}", "{doc.metadata.get("page")}"]'
        
        chunk = f"""
Source: {citation}
Content:
{doc.page_content}
"""
        formatted_chunks.append(chunk)

    return "\n\n".join(formatted_chunks)

In [48]:
def rag_pipeline(query: str):
    # Step 1: Retrieve
    candidates = vector_store.similarity_search(query, k=20)

    # Step 2: Rerank
    top_docs = rerank_documents(query, candidates, top_n=5)

    # Step 3: Format context
    context = format_context(top_docs)

    # Step 4: Build prompt
    messages = prompt.format_messages(
        context=context,
        question=query
    )

    # Step 5: LLM call
    response = llm.invoke(messages)

    return response.content

### Step 4: Answer the Test Questions
- Run your system on the 13 questions below and output answers in the specified JSON format

In [49]:
query = "What was Apple's operating income in 2023?"

answer = rag_pipeline(query)
print(answer)

According to the provided context, Apple's operating income in 2023 was $114,301 ["Apple Inc.", "47", "49.0"].


In [51]:
query = "What was Apples total revenue for the fiscal year ended September 28, 2024?"
answer = rag_pipeline(query)
print(answer)

According to the provided financial documents, Apple's total net sales (revenue) for the fiscal year ended September 28, 2024 is $391,035. This information can be found in the "CONSOLIDATED STATEMENTS OF OPERATIONS" document, specifically on page 31.0.

["Apple Inc.", "None", "31.0"]


In [52]:
question_list = [
{"question_id": 1, "question": "What was Apples total revenue for the fiscal year ended September 28, 2024?"},
{"question_id": 2, "question": "How many shares of common stock were issued and outstanding as of October 18, 2024?"},
{"question_id": 3, "question": "What is the total amount of term debt (current + non-current) reported by Apple as of September 28, 2024?"},
{"question_id": 4, "question": "On what date was Apples 10-K report for 2024 signed and filed with the SEC?"},
{"question_id": 5, "question": "Does Apple have any unresolved staff comments from the SEC as of this filing? How do you know?"},
{"question_id": 6, "question": "What was Teslas total revenue for the year ended December 31, 2023?"},
{"question_id": 7, "question": "What percentage of Teslas total revenue in 2023 came from Automotive Sales (excluding Leasing)?"},
{"question_id": 8, "question": "What is the primary reason Tesla states for being highly dependent on Elon Musk?"},
{"question_id": 9, "question": "What types of vehicles does Tesla currently produce and deliver?"},
{"question_id": 10, "question": "What is the purpose of Teslas ’lease pass-through fund arrangements’?"},
{"question_id": 11, "question": "What is Teslas stock price forecast for 2025?"},
{"question_id": 12, "question": "Who is the CFO of Apple as of 2025?"},
{"question_id": 13, "question": "What color is Teslas headquarters painted?"}
]


In [53]:
for item in question_list:
    print(f"Question ID- {item['question_id']}: {item['question']}")
    answer = rag_pipeline(item['question'])
    print(f"Answer: {answer}")
    print("-" * 50)

Question ID- 1: What was Apples total revenue for the fiscal year ended September 28, 2024?
Answer: According to the provided financial documents, Apple's total net sales (revenue) for the fiscal year ended September 28, 2024 is $391,035. This information can be found in the "CONSOLIDATED STATEMENTS OF OPERATIONS" document, specifically on page 31.0.

["Apple Inc.", "None", "31.0"]
--------------------------------------------------
Question ID- 2: How many shares of common stock were issued and outstanding as of October 18, 2024?
Answer: According to the context, ["None", "None", "1.0"] states: "15,115,823,000 shares of common stock were issued and outstanding as of October 18, 2024."
--------------------------------------------------
Question ID- 3: What is the total amount of term debt (current + non-current) reported by Apple as of September 28, 2024?
Answer: According to the provided context:

Source: ["None", "None", "33.0"]
Content:
Term debt  10,912  9,822 
Non-current liabiliti